In [ ]:
# Install all required libraries
!pip install -q accelerate transformers ftfy bitsandbytes peft torchao
!pip install git+https://github.com/huggingface/diffusers.git
!pip install -q clean-fid deepface lpips

from accelerate.utils import write_basic_config
write_basic_config()

  Cloning https://github.com/huggingface/diffusers.git to /tmp/pip-req-build-yjic02dv
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers.git /tmp/pip-req-build-yjic02dv
  Resolved https://github.com/huggingface/diffusers.git to commit 48f39c2d59e8db444cb37f91e72413a1db9a2dd6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Configuration already exists at /root/.cache/huggingface/accelerate/default_config.yaml, will not override. Run `accelerate config` manually or pass a different `save_location`.


False

In [ ]:
import os
import shutil
import json
import random
import zipfile
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPTokenizer, BitsAndBytesConfig
from diffusers import StableDiffusionPipeline, UNet2DConditionModel, DDPMScheduler
from peft import get_peft_model, LoraConfig, PeftModel
from bitsandbytes.optim import AdamW8bit

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
zip_path     = '/content/drive/MyDrive/CelebAMask-HQ.zip'
extract_path = '/content/celeba_data/'

if not os.path.exists(extract_path):
    print('Extracting to local storage...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
    print('Extraction complete.')
else:
    print('Dataset already extracted — skipping.')

Dataset already extracted — skipping.


In [ ]:
# Load the HQ↔original bridge mapping
mapping_df = pd.read_csv(
    '/content/celeba_data/CelebAMask-HQ/CelebA-HQ-to-CelebA-mapping.txt',
    sep=r'\s+', header=0
)

# Load original identity labels
identity_df = pd.read_csv(
    '/content/drive/MyDrive/identity_CelebA.txt',
    sep=r'\s+', header=None, names=['orig_file', 'person_id']
)

# Merge and build HQ filename column
merged_df = pd.merge(mapping_df, identity_df, on='orig_file')
merged_df['hq_filename'] = merged_df['idx'].apply(lambda x: f'{x}.jpg')
final_mapping = merged_df[['hq_filename', 'person_id']]

print(final_mapping.head())

  hq_filename  person_id
0       0.jpg       7423
1       1.jpg       7319
2       2.jpg       6632
3       3.jpg       3338
4       4.jpg       9178


In [ ]:
# Count HQ images per person and filter for >= 5
id_counts  = final_mapping['person_id'].value_counts()
candidates = id_counts[id_counts >= 5].index.tolist()
print(f'Total candidates with >=5 images: {len(candidates)}')

# Pick first 3 for experiments; use index 0 here
selected_ids  = candidates[:3]
target_person = selected_ids[0]
print(f'Selected Person IDs: {selected_ids}')
print(f'Using: {target_person}')

Total candidates with >=5 images: 2398
Selected Person IDs: [861, 5247, 5884]
Using: 861


In [ ]:
base_img_path = '/content/celeba_data/CelebAMask-HQ/CelebA-HQ-img/'
path_1shot    = f'/content/train_{target_person}_1shot'

os.makedirs(path_1shot, exist_ok=True)

# Get all HQ image filenames for this person
person_images = final_mapping[
    final_mapping['person_id'] == target_person
]['hq_filename'].tolist()

# Copy only 1 image for 1-shot
dst = os.path.join(path_1shot, 'img_0.jpg')
shutil.copy(os.path.join(base_img_path, person_images[0]), dst)

print(f'Folder contents : {os.listdir(path_1shot)}')
print(f'Source image    : {person_images[0]}')
print(f'Copied to       : {dst}')

Folder contents : ['metadata.jsonl', 'img_0.jpg']
Source image    : 183.jpg
Copied to       : /content/train_861_1shot/img_0.jpg


In [ ]:
def create_metadata(folder_path, trigger_phrase='a photo of sks_person'):
    metadata_file = os.path.join(folder_path, 'metadata.jsonl')
    with open(metadata_file, 'w') as f:
        for img_name in os.listdir(folder_path):
            if img_name.endswith('.jpg'):
                line = {'file_name': img_name, 'text': trigger_phrase}
                f.write(json.dumps(line) + '\n')
    print(f'Metadata written to {metadata_file}')

create_metadata(path_1shot)
print(f'Training folder: {path_1shot}')

Metadata written to /content/train_861_1shot/metadata.jsonl
Training folder: /content/train_861_1shot


In [ ]:
%%writefile qlora_train.py
import os, json, argparse, torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from transformers import CLIPTokenizer, BitsAndBytesConfig
from diffusers import StableDiffusionPipeline, UNet2DConditionModel, DDPMScheduler
from peft import get_peft_model, LoraConfig
from bitsandbytes.optim import AdamW8bit

# Args
parser = argparse.ArgumentParser()
parser.add_argument('--train_data_dir',  required=True)
parser.add_argument('--output_dir',      required=True)
parser.add_argument('--model_id',        default='runwayml/stable-diffusion-v1-5')
parser.add_argument('--resolution',      type=int,   default=512)
parser.add_argument('--batch_size',      type=int,   default=1)
parser.add_argument('--max_train_steps', type=int,   default=500)
parser.add_argument('--learning_rate',   type=float, default=1e-4)
parser.add_argument('--lora_rank',       type=int,   default=4)
args = parser.parse_args()

os.makedirs(args.output_dir, exist_ok=True)
device = torch.device('cuda')
dtype  = torch.float16

# Load UNet in 4-bit
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

unet = UNet2DConditionModel.from_pretrained(
    args.model_id,
    subfolder='unet',
    quantization_config=quant_config,
    device_map='auto',
    torch_dtype=dtype,
)

# Wrap with LoRA adapters
lora_config = LoraConfig(
    r=args.lora_rank,
    lora_alpha=args.lora_rank * 2,
    target_modules=['to_q', 'to_k', 'to_v', 'to_out.0'],
    lora_dropout=0.05,
    bias='none',
)
unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

# Load frozen pipeline components
pipe = StableDiffusionPipeline.from_pretrained(
    args.model_id,
    unet=unet,
    torch_dtype=dtype,
    safety_checker=None,
)
pipe.vae.to(device, dtype=dtype)
pipe.text_encoder.to(device, dtype=dtype)
pipe.vae.requires_grad_(False)
pipe.text_encoder.requires_grad_(False)

tokenizer       = pipe.tokenizer
noise_scheduler = DDPMScheduler.from_pretrained(args.model_id, subfolder='scheduler')

# Dataset
class SimpleDataset(Dataset):
    def __init__(self, folder, tokenizer, resolution):
        self.folder    = folder
        self.tokenizer = tokenizer
        meta_path      = os.path.join(folder, 'metadata.jsonl')
        self.items     = []
        with open(meta_path) as f:
            for line in f:
                obj = json.loads(line)
                self.items.append((obj['file_name'], obj['text']))
        self.transform = transforms.Compose([
            transforms.Resize(resolution),
            transforms.CenterCrop(resolution),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        fname, caption = self.items[idx]
        img            = Image.open(os.path.join(self.folder, fname)).convert('RGB')
        pixel_values   = self.transform(img)
        input_ids      = self.tokenizer(
            caption,
            padding='max_length',
            truncation=True,
            max_length=self.tokenizer.model_max_length,
            return_tensors='pt',
        ).input_ids[0]
        return {'pixel_values': pixel_values, 'input_ids': input_ids}

dataset    = SimpleDataset(args.train_data_dir, tokenizer, args.resolution)
dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True, pin_memory=True)

# ── 5. Optimizer: 8-bit Adam on LoRA params only ──────────────────────────────
trainable_params = [p for p in unet.parameters() if p.requires_grad]
optimizer        = AdamW8bit(trainable_params, lr=args.learning_rate)

# ── 6. Training loop ──────────────────────────────────────────────────────────
unet.train()
scaler = torch.cuda.amp.GradScaler()
step   = 0

print(f'Starting QLoRA training for {args.max_train_steps} steps...')

while step < args.max_train_steps:
    for batch in dataloader:
        if step >= args.max_train_steps:
            break

        pixel_values = batch['pixel_values'].to(device, dtype=dtype)
        input_ids    = batch['input_ids'].to(device)

        # Encode image → latents
        with torch.no_grad():
            latents = pipe.vae.encode(pixel_values).latent_dist.sample()
            latents = latents * pipe.vae.config.scaling_factor

        # Sample noise and timesteps
        noise         = torch.randn_like(latents)
        timesteps     = torch.randint(
            0, noise_scheduler.config.num_train_timesteps,
            (latents.shape[0],), device=device
        ).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        # Encode text
        with torch.no_grad():
            encoder_hidden_states = pipe.text_encoder(input_ids)[0]

        # Forward pass through 4-bit UNet with LoRA adapters active
        with torch.cuda.amp.autocast():
            noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
            loss       = torch.nn.functional.mse_loss(noise_pred, noise)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        step += 1
        if step % 50 == 0:
            print(f'  step {step}/{args.max_train_steps}  loss={loss.item():.4f}')

# ── 7. Save LoRA adapter weights only ─────────────────────────────────────────
unet.save_pretrained(args.output_dir)
print(f'QLoRA weights saved to {args.output_dir}')

Overwriting qlora_train.py


In [ ]:
output_dir = f'/content/lora_output_{target_person}_qlora'

!python qlora_train.py \
  --train_data_dir="{path_1shot}" \
  --output_dir="{output_dir}" \
  --model_id="runwayml/stable-diffusion-v1-5" \
  --resolution=256 \
  --batch_size=1 \
  --max_train_steps=500 \
  --learning_rate=1e-4 \
  --lora_rank=8

Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/usr/local/lib/python3.12/dist-packages/diffusers/utils/import_utils.py", line 41, in <module>
    _package_map = importlib_metadata.packages_distributions()  # load-once to avoid expensive calls
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/metadata/__init__.py", line 947, in packages_distributions
    for pkg in _top_level_declared(dist) or _top_level_inferred(dist):
                                            ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/metadata/__init__.py", line 959, in _top_level_inferred
    for f in always_iterable(dist.files)
                          

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

# Load 4-bit UNet base
unet = UNet2DConditionModel.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    subfolder='unet',
    quantization_config=quant_config,
    device_map='auto',
    torch_dtype=torch.float16,
)

# Load the PEFT LoRA adapters saved during training
unet = PeftModel.from_pretrained(unet, output_dir)

# Assemble inference pipeline
pipe = StableDiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    unet=unet,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe.to('cuda')
print('Pipeline loaded with QLoRA weights.')

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Expected types for unet: (<class 'diffusers.models.unets.unet_2d_condition.UNet2DConditionModel'>,), got <class 'peft.peft_model.PeftModel'>.


Pipeline loaded with QLoRA weights.


In [ ]:
fixed_prompts = [
    'A high-quality studio headshot of sks_person, facing the camera, neutral expression, plain grey background, soft even lighting, 35mm lens, realistic skin texture.',
    'A high-quality studio portrait of sks_person, upper body, looking slightly to the left, neutral expression, plain grey background, soft directional lighting from the right.',
    'A high-quality portrait of sks_person, facing the camera, big natural smile, teeth visible, blurred indoor background, warm soft lighting.',
    'A cinematic close-up of sks_person, neutral expression, dark background, strong side lighting creating shadows on half of the face, high contrast, realistic skin texture.',
    'A realistic outdoor portrait of sks_person, upper body, standing in a city street during the day, soft natural daylight, shallow depth of field, background slightly blurred.',
    'A realistic outdoor portrait of sks_person, upper body, standing in a city at sunset, warm orange light on the face, soft bokeh lights in the background.',
    'A realistic portrait of sks_person, sitting indoors near a window, soft daylight from the side, bookshelf in the background, natural colours, medium shot.',
    'A realistic portrait of sks_person, facing the camera, wearing simple glasses, neutral expression, plain light background, soft studio lighting.',
    'A realistic half-body photo of sks_person, standing, facing the camera, neutral expression, simple indoor background, soft even lighting.',
    'A realistic portrait of sks_person, varied pose and expression, random indoor or outdoor background, natural lighting, photographic style.',
]

results_dir = f'/content/generated_{target_person}'
os.makedirs(results_dir, exist_ok=True)

image_count = 0
for prompt in fixed_prompts:
    for i in range(30):
        image = pipe(prompt, num_inference_steps=30).images[0]
        image.save(f'{results_dir}/img_{image_count}.png')
        image_count += 1

print(f'Generated {image_count} images for evaluation.')

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

...

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
save_dir = f'/content/drive/MyDrive/QLoRA/Person_{target_person}'
!mkdir -p "{save_dir}"
!cp -r "{results_dir}" "{save_dir}/"
!cp -r "{output_dir}" "{save_dir}/"
print('Transfer to Google Drive complete!')

In [ ]:
from cleanfid import fid

real_dir = f'/content/real_test_{target_person}'

# Clean up any conflicting path (file or directory)
if os.path.exists(real_dir):
    if os.path.isfile(real_dir):
        os.remove(real_dir)
        print(f'Removed conflicting file: {real_dir}')
    elif os.path.isdir(real_dir):
        shutil.rmtree(real_dir)
        print(f'Removed conflicting directory: {real_dir}')

os.makedirs(real_dir, exist_ok=True)

# Copy all identity images for this person
person_imgs = final_mapping[
    final_mapping['person_id'] == target_person
]['hq_filename'].tolist()

for fn in person_imgs:
    shutil.copy(os.path.join(base_img_path, fn), real_dir)

# Pad with random faces to reach 300 total
if len(person_imgs) < 300:
    all_imgs = [f for f in os.listdir(base_img_path) if f.endswith('.jpg')]
    all_imgs = [f for f in all_imgs if f not in person_imgs]
    extras   = random.sample(all_imgs, 300 - len(person_imgs))
    for fn in extras:
        shutil.copy(os.path.join(base_img_path, fn), real_dir)

print(f'Real images built: {len(os.listdir(real_dir))}')

# Compute FID
fid_score = fid.compute_fid(results_dir, real_dir)
print(f'FID → {fid_score:.2f}')

In [ ]:
from deepface import DeepFace

def get_embedding(img_path):
    try:
        result = DeepFace.represent(img_path, model_name='ArcFace', enforce_detection=False)
        return np.array(result[0]['embedding'])
    except:
        return None

def cosine_sim(a, b):
    a = a / (np.linalg.norm(a) + 1e-8)
    b = b / (np.linalg.norm(b) + 1e-8)
    return float(np.dot(a, b))

# Reference embedding from the 1-shot training image
ref_files = [f for f in os.listdir(path_1shot) if f.lower().endswith(('.jpg', '.png'))]
ref_emb   = get_embedding(os.path.join(path_1shot, ref_files[0]))

# Compute cosine similarity for all generated images
sims = []
for fn in tqdm(os.listdir(results_dir), desc='FRS'):
    if fn.lower().endswith('.png'):
        emb = get_embedding(os.path.join(results_dir, fn))
        if emb is not None:
            sims.append(cosine_sim(emb, ref_emb))

print(f'FRS → mean={np.mean(sims):.4f}  std={np.std(sims):.4f}')

# Boxplot
plt.figure(figsize=(4, 5))
plt.boxplot(sims, labels=['1-shot'])
plt.title(f'FRS — Person {target_person} | 1-shot')
plt.ylabel('Cosine Similarity (ArcFace)')
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/QLoRA/frs_boxplot_{target_person}.png', dpi=150)
plt.show()

In [ ]:
import lpips

loss_fn = lpips.LPIPS(net='alex').cuda()

def load_tensor(img_path, size=256):
    img = Image.open(img_path).convert('RGB').resize((size, size))
    t   = transforms.ToTensor()(img) * 2 - 1
    return t.unsqueeze(0).cuda()

gen_files = [
    os.path.join(results_dir, f)
    for f in os.listdir(results_dir) if f.lower().endswith('.png')
]

rng   = np.random.default_rng(42)
idxs  = rng.choice(len(gen_files), size=(200, 2), replace=True)
dists = []

for i, j in tqdm(idxs, desc='LPIPS pairs'):
    if i == j:
        continue
    with torch.no_grad():
        d = loss_fn(load_tensor(gen_files[i]), load_tensor(gen_files[j]))
    dists.append(d.item())

print(f'LPIPS Diversity → {np.mean(dists):.4f}')

In [ ]:
# Build reference embeddings from training images
ref_files_full = [
    os.path.join(path_1shot, f)
    for f in os.listdir(path_1shot)
    if f.lower().endswith(('.jpg', '.png'))
]
ref_embs = [(fp, get_embedding(fp)) for fp in ref_files_full]
ref_embs = [(fp, e) for fp, e in ref_embs if e is not None]

# Sample 5 random generated images
gen_files = [
    os.path.join(results_dir, f)
    for f in os.listdir(results_dir) if f.lower().endswith('.png')
]
sampled = random.sample(gen_files, min(5, len(gen_files)))

fig, axes = plt.subplots(len(sampled), 2, figsize=(4, 2.5 * len(sampled)))

for row, gen_path in enumerate(sampled):
    gen_emb = get_embedding(gen_path)
    if gen_emb is None:
        continue

    similarities = [(fp, float(cosine_sim(gen_emb, e))) for fp, e in ref_embs]
    nn_path, nn_sim = max(similarities, key=lambda x: x[1])

    axes[row][0].imshow(Image.open(gen_path).resize((128, 128)))
    axes[row][0].set_title('Generated', fontsize=8)
    axes[row][0].axis('off')

    axes[row][1].imshow(Image.open(nn_path).resize((128, 128)))
    axes[row][1].set_title(f'Nearest ref\nsim={nn_sim:.3f}', fontsize=8)
    axes[row][1].axis('off')

plt.suptitle(
    f'NN Memorization Check — Person {target_person} | 1-shot\n'
    f'(high sim = possible memorization, low sim = genuine generation)',
    fontsize=9
)
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/QLoRA/Memorization_check_{target_person}.png', dpi=150)
plt.show()